# LJ Dev Commerce - Phase 4

# XII. Inventory ETL


# =========================================================
# 1.1 Setup
# =========================================================

This notebook performs the approved ETL workflow for the Inventory current-stock dataset.

Workflow: Raw Dataset → Clean CSV → Database-Ready Dataset → PostgreSQL.

The raw Inventory dataset is never modified directly.


In [1]:
# =========================================================
# 1.1 Setup
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
# Project path handling is configured in the dataset load cell.


## 1.2 Load and Preserve Inventory Raw Dataset

Load the untouched Inventory current-stock source dataset and preserve an original copy for validation.


In [2]:
# =========================================================
# 1.2 Load and Preserve Inventory Raw Dataset
# =========================================================

inventory_raw_path = Path(
    r"C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics"
    r"\data\04_Zoho_Inventory\zoho_inventory_current_stock.csv"
)

inventory_raw = pd.read_csv(inventory_raw_path)
inventory_raw_original = inventory_raw.copy(deep=True)

print("Inventory raw dataset loaded successfully.")
print("Rows:", len(inventory_raw))
print("Columns:", len(inventory_raw.columns))
print("Path:", inventory_raw_path)


Inventory raw dataset loaded successfully.
Rows: 19
Columns: 13
Path: C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\zoho_inventory_current_stock.csv


## 1.3 Initial Raw Data Inspection

Inspect the untouched Inventory current-stock dataset before any cleaning or transformation.


In [3]:
# =========================================================
# 1.3 Initial Raw Data Inspection
# =========================================================

print("Inventory Dataset shape:")
print(inventory_raw.shape)
print("\nInventory Column names:")
print(inventory_raw.columns.tolist())
print("\nInventory Current pandas data types:")
print(inventory_raw.dtypes)
print("\nInventory Missing values:")
print(inventory_raw.isnull().sum())
print("\nInventory Sample raw records:")
display(inventory_raw.head())


Inventory Dataset shape:
(19, 13)

Inventory Column names:
['InventoryRecordID', 'ProductCode', 'WarehouseCode', 'OnHandQty', 'ReorderPoint', 'ReservedQty', 'AvailableQty', 'StockStatus', 'LastUpdatedAt', 'CreatedOn', 'CreatedByUser', 'ModifiedOn', 'ModifiedByUser']

Inventory Current pandas data types:
InventoryRecordID    object
ProductCode          object
WarehouseCode        object
OnHandQty             int64
ReorderPoint          int64
ReservedQty           int64
AvailableQty          int64
StockStatus          object
LastUpdatedAt        object
CreatedOn            object
CreatedByUser        object
ModifiedOn           object
ModifiedByUser       object
dtype: object

Inventory Missing values:
InventoryRecordID    0
ProductCode          0
WarehouseCode        0
OnHandQty            0
ReorderPoint         0
ReservedQty          0
AvailableQty         0
StockStatus          0
LastUpdatedAt        0
CreatedOn            0
CreatedByUser        0
ModifiedOn           0
ModifiedByUser

,InventoryRecordID,ProductCode,WarehouseCode,OnHandQty,ReorderPoint,ReservedQty,AvailableQty,StockStatus,LastUpdatedAt,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,IN001,P0001,WH001,25,10,2,23,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
1,IN002,P0002,WH001,30,10,3,27,in stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
2,IN003,P0003,WH001,12,10,1,11,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
3,IN004,P0004,WH002,40,10,4,36,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
4,IN005,P0005,WH001,18,10,1,17,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin


## 1.4 Data Profiling

Profile the untouched Inventory dataset using the reusable project profiler.


In [4]:
# =========================================================
# 1.4 Data Profiling
# =========================================================

import sys
import importlib

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from profiler import data_profiler_v1 as profiler
importlib.reload(profiler)

print("Profiler loaded successfully:")
print(profiler.__file__)
print("\nProfiler configuration:")
print(profiler.DEFAULT_CONFIG)


Profiler loaded successfully:
c:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\profiler\data_profiler_v1.py

Profiler configuration:
{'date_detection_threshold': 0.8, 'numeric_detection_threshold': 0.8, 'email_detection_threshold': 0.8, 'phone_detection_threshold': 0.8, 'categorical_unique_ratio': 0.2, 'outlier_iqr_multiplier': 1.5, 'required_columns': [], 'unique_columns': [], 'non_negative_columns': []}


In [5]:
inventory_profile_results = profiler.profile_dataset(inventory_raw)
print("\nInventory profiling completed successfully.")
print("\nInventory profiler sections:")
print(list(inventory_profile_results.keys()))



Inventory profiling completed successfully.

Inventory profiler sections:
['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration']


## 1.5 Review Profiling Results

Review the complete Inventory profiling output before making transformation decisions.


In [6]:
print("\n" + "=" * 70)
print("INVENTORY PROFILING RESULTS")
print("=" * 70)
for section_name, section_result in inventory_profile_results.items():
    print("\n" + "-" * 70)
    print(section_name.upper())
    print("-" * 70)
    print(section_result)



INVENTORY PROFILING RESULTS

----------------------------------------------------------------------
FIELD_TYPES
----------------------------------------------------------------------
InventoryRecordID     identifier
ProductCode           identifier
WarehouseCode         identifier
OnHandQty                numeric
ReorderPoint             numeric
ReservedQty              numeric
AvailableQty             numeric
StockStatus          categorical
LastUpdatedAt               date
CreatedOn                   date
CreatedByUser        categorical
ModifiedOn                  date
ModifiedByUser       categorical
Name: DetectedFieldType, dtype: object

----------------------------------------------------------------------
DETECTION_DETAILS
----------------------------------------------------------------------
{'InventoryRecordID': {'DetectedType': 'identifier', 'Confidence': 'Medium', 'Evidence': 'Column name suggests identifier semantics'}, 'ProductCode': {'DetectedType': 'identifier', 'Confi

## 1.6 Review Profiler Issues

Review the profiler's detected Inventory issues separately and classify them before transformation.


In [7]:
inventory_issues = inventory_profile_results["issues"]
print("\n" + "=" * 70)
print("INVENTORY PROFILER ISSUES")
print("=" * 70)
display(inventory_issues)



INVENTORY PROFILER ISSUES


,Column,IssueType,Severity,Count,Description
0,ProductCode,DuplicateIdentifier,High,4,Repeated identifier values detected.
1,WarehouseCode,DuplicateIdentifier,High,16,Repeated identifier values detected.


## 1.7 Independent Analyst Review

Perform the systematic independent analyst review required by the locked ETL workflow.


In [8]:
# =========================================================
# 1.7 Independent Analyst Review
# =========================================================

source_df = inventory_raw

print("1. DATASET STRUCTURE AND GRAIN")
print("=" * 80)
print(f"Rows: {source_df.shape[0]}")
print(f"Columns: {source_df.shape[1]}")
print("Working assumption: One row represents the current inventory record for one product in one warehouse.")

print("\n\n2. MISSING VALUES AND BLANK VALUES")
missing_values = source_df.isna().sum()
blank_values = pd.Series({c:(source_df[c].astype("string").str.strip().eq("").sum() if source_df[c].dtype == "object" else 0) for c in source_df.columns})
display(pd.DataFrame({"MissingValues":missing_values,"BlankValues":blank_values}))

print("\n\n3. EXACT DUPLICATE ROWS")
exact_duplicate_count = source_df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)
if exact_duplicate_count > 0: display(source_df[source_df.duplicated(keep=False)])

print("\n\n4. PRIMARY KEY CANDIDATE AND BUSINESS KEY UNIQUENESS")
identifier_columns=["InventoryRecordID","ProductCode","WarehouseCode"]
identifier_review=pd.DataFrame({"Column":identifier_columns,"NullCount":[source_df[c].isna().sum() for c in identifier_columns],"BlankCount":[source_df[c].astype("string").str.strip().eq("").sum() for c in identifier_columns],"DuplicateCount":[source_df[c].duplicated().sum() for c in identifier_columns],"UniqueValues":[source_df[c].nunique(dropna=True) for c in identifier_columns]})
display(identifier_review)

print("\n\n5. INVENTORY IDENTIFIER COLLISION REVIEW")
dups=source_df[source_df["InventoryRecordID"].duplicated(keep=False)]
print("Duplicate InventoryRecordID records:",len(dups))
if not dups.empty: display(dups)

print("\n\n6. PRODUCT / WAREHOUSE BUSINESS KEY REVIEW")
business_key_dups=source_df[source_df.duplicated(subset=["ProductCode","WarehouseCode"],keep=False)]
print("Duplicate ProductCode + WarehouseCode records:",len(business_key_dups))
if not business_key_dups.empty: display(business_key_dups[["InventoryRecordID","ProductCode","WarehouseCode"]])

print("\n\n7. LEADING AND TRAILING WHITESPACE REVIEW")
text_columns=source_df.select_dtypes(include="object").columns
whitespace_findings=[]
for column in text_columns:
    mask=source_df[column].astype("string") != source_df[column].astype("string").str.strip()
    count=mask.sum()
    if count>0:
        whitespace_findings.append({"Column":column,"WhitespaceRecordCount":count})
        display(source_df.loc[mask,[column]])
if whitespace_findings: display(pd.DataFrame(whitespace_findings))
else: print("No leading or trailing whitespace found.")

print("\n\n8. STOCK STATUS CONSISTENCY REVIEW")
print("Raw StockStatus values:")
display(source_df["StockStatus"].value_counts(dropna=False).sort_index().rename("RecordCount").to_frame())
print("\nObserved ProductCode count:",source_df["ProductCode"].nunique())
print("Observed WarehouseCode count:",source_df["WarehouseCode"].nunique())

print("\n\n9. INVENTORY QUANTITY VALIDITY REVIEW")
for column in ["OnHandQty","ReorderPoint","ReservedQty","AvailableQty"]:
    print(f"{column}: dtype={source_df[column].dtype}, nulls={source_df[column].isna().sum()}, negatives={(source_df[column]<0).sum()}")
calculated_available=source_df["OnHandQty"]-source_df["ReservedQty"]
available_mismatch=(source_df["AvailableQty"]!=calculated_available)
print("AvailableQty = OnHandQty - ReservedQty mismatches:",available_mismatch.sum())
print("\nReorderPoint values:")
display(source_df["ReorderPoint"].value_counts(dropna=False).sort_index().rename("RecordCount").to_frame())

print("\n\n10. REORDER-POINT / STOCK-STATUS BUSINESS RULE REVIEW")
below_reorder=source_df["AvailableQty"] < source_df["ReorderPoint"]
at_or_above_reorder=source_df["AvailableQty"] >= source_df["ReorderPoint"]
print("Records below reorder point:",below_reorder.sum())
print("Records at/above reorder point:",at_or_above_reorder.sum())
status_review=source_df[["InventoryRecordID","AvailableQty","ReorderPoint","StockStatus"]].copy()
status_review["BelowReorderPoint"]=status_review["AvailableQty"] < status_review["ReorderPoint"]
display(status_review)

print("\n\n11. DATE VALIDITY AND DATATYPE REVIEW")
for column in ["LastUpdatedAt","CreatedOn","ModifiedOn"]:
    parsed=pd.to_datetime(source_df[column],errors="coerce")
    invalid=(parsed.isna() & source_df[column].notna()).sum()
    print(f"{column}: pandas dtype={source_df[column].dtype}, invalid populated dates={invalid}")

print("\n\n12. DATE RELATIONSHIP REVIEW")
last_updated=pd.to_datetime(source_df["LastUpdatedAt"],errors="coerce")
created=pd.to_datetime(source_df["CreatedOn"],errors="coerce")
modified=pd.to_datetime(source_df["ModifiedOn"],errors="coerce")
print("ModifiedOn before CreatedOn:",(modified.notna() & created.notna() & (modified<created)).sum())
print("LastUpdatedAt before CreatedOn:",(last_updated.notna() & created.notna() & (last_updated<created)).sum())

print("\n\n13. SOURCE-TO-TARGET DATATYPE PREPARATION REVIEW")
datatype_review=pd.DataFrame([
["InventoryRecordID",str(source_df["InventoryRecordID"].dtype),"Text","Retain"],
["ProductCode",str(source_df["ProductCode"].dtype),"Text","Retain"],
["WarehouseCode",str(source_df["WarehouseCode"].dtype),"Text","Retain"],
["OnHandQty",str(source_df["OnHandQty"].dtype),"Integer","Retain"],
["ReorderPoint",str(source_df["ReorderPoint"].dtype),"Integer","Retain"],
["ReservedQty",str(source_df["ReservedQty"].dtype),"Integer","Retain"],
["AvailableQty",str(source_df["AvailableQty"].dtype),"Integer","Retain"],
["StockStatus",str(source_df["StockStatus"].dtype),"Text","Standardize exact value 'in stock' to 'In Stock'"],
["LastUpdatedAt",str(source_df["LastUpdatedAt"].dtype),"Datetime","Convert"],
["CreatedOn",str(source_df["CreatedOn"].dtype),"Datetime","Convert"],
["CreatedByUser",str(source_df["CreatedByUser"].dtype),"Text","Retain"],
["ModifiedOn",str(source_df["ModifiedOn"].dtype),"Datetime","Convert"],
["ModifiedByUser",str(source_df["ModifiedByUser"].dtype),"Text","Retain"]
],columns=["SourceColumn","CurrentPandasDtype","TargetType","Preparation"])
display(datatype_review)

print("\n\n14. REFERENTIAL READINESS REVIEW")
print("ProductCode references the Product master through product_id.")
print("WarehouseCode references the Warehouse master through warehouse_id.")
print("Actual FK readiness is validated later against PostgreSQL.")


1. DATASET STRUCTURE AND GRAIN
Rows: 19
Columns: 13
Working assumption: One row represents the current inventory record for one product in one warehouse.


2. MISSING VALUES AND BLANK VALUES


,MissingValues,BlankValues
InventoryRecordID,0,0
ProductCode,0,0
WarehouseCode,0,0
OnHandQty,0,0
ReorderPoint,0,0
ReservedQty,0,0
AvailableQty,0,0
StockStatus,0,0
LastUpdatedAt,0,0
CreatedOn,0,0




3. EXACT DUPLICATE ROWS
Exact duplicate rows: 0


4. PRIMARY KEY CANDIDATE AND BUSINESS KEY UNIQUENESS


,Column,NullCount,BlankCount,DuplicateCount,UniqueValues
0,InventoryRecordID,0,0,0,19
1,ProductCode,0,0,4,15
2,WarehouseCode,0,0,16,3




5. INVENTORY IDENTIFIER COLLISION REVIEW
Duplicate InventoryRecordID records: 0


6. PRODUCT / WAREHOUSE BUSINESS KEY REVIEW
Duplicate ProductCode + WarehouseCode records: 0


7. LEADING AND TRAILING WHITESPACE REVIEW


...



10. REORDER-POINT / STOCK-STATUS BUSINESS RULE REVIEW
Records below reorder point: 3
Records at/above reorder point: 16


,InventoryRecordID,AvailableQty,ReorderPoint,StockStatus,BelowReorderPoint
0,IN001,23,10,In Stock,False
1,IN002,27,10,in stock,False
2,IN003,11,10,In Stock,False
3,IN004,36,10,In Stock,False
4,IN005,17,10,In Stock,False
5,IN006,20,10,In Stock,False
6,IN007,15,10,In Stock,False
7,IN008,18,10,In Stock,False
8,IN009,32,10,In Stock,False
9,IN010,9,10,Low Stock,True




11. DATE VALIDITY AND DATATYPE REVIEW
LastUpdatedAt: pandas dtype=object, invalid populated dates=0
CreatedOn: pandas dtype=object, invalid populated dates=0
ModifiedOn: pandas dtype=object, invalid populated dates=0


12. DATE RELATIONSHIP REVIEW
ModifiedOn before CreatedOn: 0
LastUpdatedAt before CreatedOn: 0


13. SOURCE-TO-TARGET DATATYPE PREPARATION REVIEW


,SourceColumn,CurrentPandasDtype,TargetType,Preparation
0,InventoryRecordID,object,Text,Retain
1,ProductCode,object,Text,Retain
2,WarehouseCode,object,Text,Retain
3,OnHandQty,int64,Integer,Retain
4,ReorderPoint,int64,Integer,Retain
5,ReservedQty,int64,Integer,Retain
6,AvailableQty,int64,Integer,Retain
7,StockStatus,object,Text,Standardize exact value 'in stock' to 'In Stock'
8,LastUpdatedAt,object,Datetime,Convert
9,CreatedOn,object,Datetime,Convert




14. REFERENTIAL READINESS REVIEW
ProductCode references the Product master through product_id.
WarehouseCode references the Warehouse master through warehouse_id.
Actual FK readiness is validated later against PostgreSQL.


## 1.8 Findings and Transformation Decisions

Compare profiler findings, independent analyst findings, and the approved Inventory Data Dictionary/business rules. Only approved transformations proceed to Section 2.


In [15]:
print("=" * 70)
print("INVENTORY FINDINGS AND TRANSFORMATION DECISIONS")
print("=" * 70)

inventory_decisions = [
    {
        "Finding": "InventoryRecordID uniqueness",
        "AffectedRecords": 19,
        "Decision": "Valid value — retain unchanged",
        "ApprovedAction": "Use as inventory_id",
        "Reason": "Approved primary identifier."
    },
    {
        "Finding": "ProductCode + WarehouseCode uniqueness",
        "AffectedRecords": 19,
        "Decision": "Valid business grain — retain unchanged",
        "ApprovedAction": "Keep values unchanged",
        "Reason": "One inventory record represents one product in one warehouse."
    },
    {
        "Finding": "No exact duplicate rows",
        "AffectedRecords": 0,
        "Decision": "Valid value — retain unchanged",
        "ApprovedAction": "Keep all 19 records",
        "Reason": "No duplicate records identified."
    },
    {
        "Finding": "StockStatus whitespace and casing/value variation",
        "AffectedRecords": 1,
        "Decision": "Approved standardization",
        "ApprovedAction": "Trim leading/trailing whitespace and convert the observed 'in stock' value to 'In Stock'",
        "Reason": "Standardize the observed whitespace and casing/value variation only; no other StockStatus values are changed."
    },
    {
        "Finding": "LastUpdatedAt object dtype",
        "AffectedRecords": 19,
        "Decision": "Safe deterministic transformation",
        "ApprovedAction": "Convert to datetime",
        "Reason": "Target last_stock_update is Datetime and Required."
    },
    {
        "Finding": "CreatedOn object dtype",
        "AffectedRecords": 19,
        "Decision": "Safe deterministic transformation",
        "ApprovedAction": "Convert to datetime",
        "Reason": "Target created_date is Datetime and Required."
    },
    {
        "Finding": "ModifiedOn object dtype",
        "AffectedRecords": 19,
        "Decision": "Safe deterministic transformation",
        "ApprovedAction": "Convert to datetime",
        "Reason": "Target updated_date is Datetime and Required."
    },
    {
        "Finding": "Inventory quantity values and relationship",
        "AffectedRecords": 19,
        "Decision": "Valid value — retain unchanged",
        "ApprovedAction": "Keep source quantities unchanged",
        "Reason": "AvailableQty equals OnHandQty minus ReservedQty for all records."
    },
    {
        "Finding": "ReorderPoint values",
        "AffectedRecords": 19,
        "Decision": "Valid value — retain unchanged",
        "ApprovedAction": "Keep values unchanged",
        "Reason": "No approved transformation; values are valid source business data."
    },
    {
        "Finding": "Inventory identifiers and audit fields",
        "AffectedRecords": 19,
        "Decision": "Valid value — retain unchanged",
        "ApprovedAction": "Keep values unchanged",
        "Reason": "No approved cleaning requirement identified."
    },
    {
        "Finding": "Product / Warehouse references",
        "AffectedRecords": 19,
        "Decision": "Requires relationship validation later",
        "ApprovedAction": "Validate against parent tables in PostgreSQL",
        "Reason": "Both are approved foreign keys."
    }
]

inventory_decisions_df = pd.DataFrame(inventory_decisions)

display(inventory_decisions_df)

print("\n" + "=" * 70)
print("TRANSFORMATION DECISIONS")
print("=" * 70)

print("1. Standardize StockStatus by trimming leading/trailing whitespace")
print("   and converting the observed 'in stock' value to 'In Stock'.")
print("2. Convert LastUpdatedAt from object to datetime.")
print("3. Convert CreatedOn from object to datetime.")
print("4. Convert ModifiedOn from object to datetime.")
print("5. Preserve all other source values and columns unchanged.")
print("6. Preserve the AvailableQty = OnHandQty - ReservedQty relationship.")
print("7. Validate Product and Warehouse foreign-key readiness later")
print("   against PostgreSQL parent tables.")

INVENTORY FINDINGS AND TRANSFORMATION DECISIONS


,Finding,AffectedRecords,Decision,ApprovedAction,Reason
0,InventoryRecordID uniqueness,19,Valid value — retain unchanged,Use as inventory_id,Approved primary identifier.
1,ProductCode + WarehouseCode uniqueness,19,Valid business grain — retain unchanged,Keep values unchanged,One inventory record represents one product in one warehouse.
2,No exact duplicate rows,0,Valid value — retain unchanged,Keep all 19 records,No duplicate records identified.
3,StockStatus whitespace and casing/value variation,1,Approved standardization,Trim leading/trailing whitespace and convert the observed 'in stock' value to 'In Stock',Standardize the observed whitespace and casing/value variation only; no other StockStatus values are changed.
4,LastUpdatedAt object dtype,19,Safe deterministic transformation,Convert to datetime,Target last_stock_update is Datetime and Required.
5,CreatedOn object dtype,19,Safe deterministic transformation,Convert to datetime,Target created_date is Datetime and Required.
6,ModifiedOn object dtype,19,Safe deterministic transformation,Convert to datetime,Target updated_date is Datetime and Required.
7,Inventory quantity values and relationship,19,Valid value — retain unchanged,Keep source quantities unchanged,AvailableQty equals OnHandQty minus ReservedQty for all records.
8,ReorderPoint values,19,Valid value — retain unchanged,Keep values unchanged,No approved transformation; values are valid source business data.
9,Inventory identifiers and audit fields,19,Valid value — retain unchanged,Keep values unchanged,No approved cleaning requirement identified.



TRANSFORMATION DECISIONS
1. Standardize StockStatus by trimming leading/trailing whitespace
   and converting the observed 'in stock' value to 'In Stock'.
2. Convert LastUpdatedAt from object to datetime.
3. Convert CreatedOn from object to datetime.
4. Convert ModifiedOn from object to datetime.
5. Preserve all other source values and columns unchanged.
6. Preserve the AvailableQty = OnHandQty - ReservedQty relationship.
7. Validate Product and Warehouse foreign-key readiness later
   against PostgreSQL parent tables.


# =========================================================
# SECTION 2 — DATA TRANSFORMATION
# =========================================================

## 2.1 Create Clean Working Dataset

Create a separate working copy of the raw Inventory dataset before applying any approved transformations.

The raw Inventory dataset must remain unchanged throughout the ETL process.

All transformations will be applied only to the clean working dataset.

The Inventory source dataset will remain as a single working dataset during this step.


In [10]:
inventory_clean=inventory_raw.copy(deep=True)
print("Inventory Clean DataFrame created.")
print("Rows:",len(inventory_clean))
print("Columns:",len(inventory_clean.columns))


Inventory Clean DataFrame created.
Rows: 19
Columns: 13


## 2.2 Apply Approved Transformations

Apply only the approved Inventory transformations documented in Section 1.8.


In [16]:
inventory_clean["StockStatus"] = (
    inventory_clean["StockStatus"]
    .str.strip()
    .replace({"in stock": "In Stock"})
)

for column in ["LastUpdatedAt", "CreatedOn", "ModifiedOn"]:
    inventory_clean[column] = pd.to_datetime(
        inventory_clean[column],
        errors="raise"
    )

print("Approved Inventory transformations applied.")

Approved Inventory transformations applied.


## 3.1 Transformation Validation

Validate every approved Inventory transformation from Section 1.8.

Checks include StockStatus standardization, datetime conversion, preservation of approved quantity relationships, and preservation of all other approved Inventory values.


In [17]:
stock_status_valid="in stock" not in inventory_clean["StockStatus"].tolist() and set(inventory_clean["StockStatus"].dropna().unique()).issubset({"In Stock","Low Stock"})
datetime_valid=all(pd.api.types.is_datetime64_any_dtype(inventory_clean[c]) for c in ["LastUpdatedAt","CreatedOn","ModifiedOn"])
quantity_relationship_valid=(inventory_clean["AvailableQty"]==inventory_clean["OnHandQty"]-inventory_clean["ReservedQty"]).all()
print("Inventory Transformation Validation")
print("StockStatus standardization valid:",stock_status_valid)
print("Datetime fields valid:",datetime_valid)
print("Inventory quantity relationship preserved:",quantity_relationship_valid)
transformation_validation_passed=all([stock_status_valid,datetime_valid,quantity_relationship_valid])
print("Overall transformation validation passed:",transformation_validation_passed)


Inventory Transformation Validation
StockStatus standardization valid: True
Datetime fields valid: True
Inventory quantity relationship preserved: True
Overall transformation validation passed: True


## 3.2 Validate Data Preservation

Validate that the approved Inventory transformations did not unintentionally change unrelated values or remove records.


In [18]:
preserved_columns=["InventoryRecordID","ProductCode","WarehouseCode","OnHandQty","ReorderPoint","ReservedQty","AvailableQty","CreatedByUser","ModifiedByUser"]
row_count_preserved=len(inventory_clean)==len(inventory_raw)
column_structure_preserved=inventory_clean.columns.tolist()==inventory_raw.columns.tolist()
preservation_results={c:inventory_clean[c].equals(inventory_raw[c]) for c in preserved_columns}
preserved_values_valid=all(preservation_results.values())
inventory_ids_preserved=inventory_clean["InventoryRecordID"].tolist()==inventory_raw["InventoryRecordID"].tolist()
data_preservation_passed=all([row_count_preserved,column_structure_preserved,preserved_values_valid,inventory_ids_preserved])
display(pd.Series(preservation_results,name="Preserved"))
print("Row count preserved:",row_count_preserved)
print("Column structure preserved:",column_structure_preserved)
print("Inventory IDs preserved:",inventory_ids_preserved)
print("Unchanged values preserved:",preserved_values_valid)
print("Overall data preservation validation passed:",data_preservation_passed)


InventoryRecordID    True
ProductCode          True
WarehouseCode        True
OnHandQty            True
ReorderPoint         True
ReservedQty          True
AvailableQty         True
CreatedByUser        True
ModifiedByUser       True
Name: Preserved, dtype: bool

Row count preserved: True
Column structure preserved: True
Inventory IDs preserved: True
Unchanged values preserved: True
Overall data preservation validation passed: True


# =========================================================
# SECTION 4 — CLEAN CSV
# =========================================================


In [19]:
# =========================================================
# 4.1 Export Clean Inventory Dataset
# =========================================================

inventory_clean_dir=Path(r"C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\clean")
inventory_clean_dir.mkdir(parents=True,exist_ok=True)
inventory_clean_path=inventory_clean_dir / "zoho_inventory_current_stock_clean.csv"
inventory_clean.to_csv(inventory_clean_path,index=False)
print("Clean Inventory CSV exported successfully.")
print("Path:",inventory_clean_path)


Clean Inventory CSV exported successfully.
Path: C:\JEP\DATA ANALYST PORTFOLIO\ecommerce-data-analytics\data\04_Zoho_Inventory\clean\zoho_inventory_current_stock_clean.csv


## 4.2 Verify Exported Clean Inventory CSV

Reload the exported Clean CSV and verify row count and column structure against the validated Clean DataFrame.


In [20]:
inventory_clean_csv=pd.read_csv(inventory_clean_path,parse_dates=["LastUpdatedAt","CreatedOn","ModifiedOn"])
row_count_match=len(inventory_clean_csv)==len(inventory_clean)
column_structure_match=inventory_clean_csv.columns.tolist()==inventory_clean.columns.tolist()
print("Inventory Clean CSV read-back successful.")
print("Rows:",len(inventory_clean_csv))
print("Columns:",len(inventory_clean_csv.columns))
print("Row count matches:",row_count_match)
print("Column structure matches:",column_structure_match)
display(inventory_clean_csv.head())


Inventory Clean CSV read-back successful.
Rows: 19
Columns: 13
Row count matches: True
Column structure matches: True


,InventoryRecordID,ProductCode,WarehouseCode,OnHandQty,ReorderPoint,ReservedQty,AvailableQty,StockStatus,LastUpdatedAt,CreatedOn,CreatedByUser,ModifiedOn,ModifiedByUser
0,IN001,P0001,WH001,25,10,2,23,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
1,IN002,P0002,WH001,30,10,3,27,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
2,IN003,P0003,WH001,12,10,1,11,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
3,IN004,P0004,WH002,40,10,4,36,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
4,IN005,P0005,WH001,18,10,1,17,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin


# =========================================================
# SECTION 5 — DATABASE READY
# =========================================================


In [21]:
# =========================================================
# 5.1 Define Source → Target Mapping
# =========================================================

inventory_mapping=pd.DataFrame([
["InventoryRecordID","inventory_id","TEXT","Required","PK",""],
["ProductCode","product_id","TEXT","Required","FK","commerce.product.product_id"],
["WarehouseCode","warehouse_id","TEXT","Required","FK","commerce.warehouse.warehouse_id"],
["OnHandQty","current_stock","INTEGER","Required","",""],
["ReorderPoint","reorder_level","INTEGER","Required","",""],
["ReservedQty","reserved_stock","INTEGER","Required","",""],
["AvailableQty","available_stock","INTEGER","Required","",""],
["StockStatus","inventory_status","TEXT","Required","","Standardize exact source value 'in stock' to 'In Stock'"],
["LastUpdatedAt","last_stock_update","TIMESTAMP","Required","",""],
["CreatedOn","created_date","TIMESTAMP","Required","",""],
["CreatedByUser","created_by","TEXT","Required","",""],
["ModifiedOn","updated_date","TIMESTAMP","Required","",""],
["ModifiedByUser","updated_by","TEXT","Required","",""]
],columns=["Source Column","Target Column","Target Datatype","Required","Key Role","Reference / Rule"])
display(inventory_mapping)


,Source Column,Target Column,Target Datatype,Required,Key Role,Reference / Rule
0,InventoryRecordID,inventory_id,TEXT,Required,PK,
1,ProductCode,product_id,TEXT,Required,FK,commerce.product.product_id
2,WarehouseCode,warehouse_id,TEXT,Required,FK,commerce.warehouse.warehouse_id
3,OnHandQty,current_stock,INTEGER,Required,,
4,ReorderPoint,reorder_level,INTEGER,Required,,
5,ReservedQty,reserved_stock,INTEGER,Required,,
6,AvailableQty,available_stock,INTEGER,Required,,
7,StockStatus,inventory_status,TEXT,Required,,Standardize exact source value 'in stock' to 'In Stock'
8,LastUpdatedAt,last_stock_update,TIMESTAMP,Required,,
9,CreatedOn,created_date,TIMESTAMP,Required,,


## 5.2 Create Database-Ready Inventory Dataset

Create the Database-Ready Inventory dataset from the verified exported Clean CSV, not directly from the in-memory Clean DataFrame.


In [22]:
inventory_db_source=pd.read_csv(inventory_clean_path,parse_dates=["LastUpdatedAt","CreatedOn","ModifiedOn"])
inventory_db_ready=inventory_db_source.rename(columns={"InventoryRecordID":"inventory_id","ProductCode":"product_id","WarehouseCode":"warehouse_id","OnHandQty":"current_stock","ReorderPoint":"reorder_level","ReservedQty":"reserved_stock","AvailableQty":"available_stock","StockStatus":"inventory_status","LastUpdatedAt":"last_stock_update","CreatedOn":"created_date","CreatedByUser":"created_by","ModifiedOn":"updated_date","ModifiedByUser":"updated_by"})[["inventory_id","product_id","warehouse_id","current_stock","reorder_level","reserved_stock","available_stock","inventory_status","last_stock_update","created_date","created_by","updated_date","updated_by"]].copy()
print("Inventory Database-Ready dataset created from exported Clean CSV.")
print("Rows:",len(inventory_db_ready))
print("Columns:",len(inventory_db_ready.columns))
display(inventory_db_ready.head())


Inventory Database-Ready dataset created from exported Clean CSV.
Rows: 19
Columns: 13


,inventory_id,product_id,warehouse_id,current_stock,reorder_level,reserved_stock,available_stock,inventory_status,last_stock_update,created_date,created_by,updated_date,updated_by
0,IN001,P0001,WH001,25,10,2,23,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
1,IN002,P0002,WH001,30,10,3,27,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
2,IN003,P0003,WH001,12,10,1,11,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
3,IN004,P0004,WH002,40,10,4,36,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
4,IN005,P0005,WH001,18,10,1,17,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin


## 5.3 Database-Ready Validation

Validate the Inventory Database-Ready dataset against the approved target columns, required fields, primary key, datatypes, mapping completeness, foreign-key readiness, and preserved source values.


In [23]:
expected_target_columns=["inventory_id","product_id","warehouse_id","current_stock","reorder_level","reserved_stock","available_stock","inventory_status","last_stock_update","created_date","created_by","updated_date","updated_by"]
target_columns_match=inventory_db_ready.columns.tolist()==expected_target_columns
row_count_match=len(inventory_db_ready)==len(inventory_clean_csv)
required_fields=expected_target_columns
required_null_counts=inventory_db_ready[required_fields].isnull().sum()
required_fields_valid=required_null_counts.sum()==0
primary_key_valid=inventory_db_ready["inventory_id"].is_unique and inventory_db_ready["inventory_id"].notna().all()
datetime_fields=["last_stock_update","created_date","updated_date"]
datetime_fields_valid=all(pd.api.types.is_datetime64_any_dtype(inventory_db_ready[c]) for c in datetime_fields)
numeric_fields=["current_stock","reorder_level","reserved_stock","available_stock"]
numeric_fields_valid=all(pd.api.types.is_integer_dtype(inventory_db_ready[c]) for c in numeric_fields)
mapping_complete=set(inventory_mapping["Source Column"])==set(inventory_clean_csv.columns)
status_values_valid=set(inventory_db_ready["inventory_status"].dropna().unique()).issubset({"In Stock","Low Stock"})
quantity_relationship_valid=(inventory_db_ready["available_stock"]==inventory_db_ready["current_stock"]-inventory_db_ready["reserved_stock"]).all()
display(required_null_counts)
print("Target columns match:",target_columns_match)
print("Row count match:",row_count_match)
print("Required fields valid:",required_fields_valid)
print("Primary key valid:",primary_key_valid)
print("Datetime fields valid:",datetime_fields_valid)
print("Numeric fields valid:",numeric_fields_valid)
print("Mapping complete:",mapping_complete)
print("Inventory status values valid:",status_values_valid)
print("Quantity relationship valid:",quantity_relationship_valid)
database_ready_validation_passed=all([target_columns_match,row_count_match,required_fields_valid,primary_key_valid,datetime_fields_valid,numeric_fields_valid,mapping_complete,status_values_valid,quantity_relationship_valid])
print("Inventory Database-Ready validation passed:",database_ready_validation_passed)


inventory_id         0
product_id           0
warehouse_id         0
current_stock        0
reorder_level        0
reserved_stock       0
available_stock      0
inventory_status     0
last_stock_update    0
created_date         0
created_by           0
updated_date         0
updated_by           0
dtype: int64

Target columns match: True
Row count match: True
Required fields valid: True
Primary key valid: True
Datetime fields valid: True
Numeric fields valid: True
Mapping complete: True
Inventory status values valid: True
Quantity relationship valid: True
Inventory Database-Ready validation passed: True


# =========================================================
# SECTION 6 — PostgreSQL
# =========================================================


## 6.1 Connect

Establish a PostgreSQL connection for the Inventory ETL process.


In [24]:
import psycopg2
from getpass import getpass

DB_HOST="localhost"
DB_PORT="5432"
DB_NAME="lj_dev_commerce"
DB_USER="postgres"
DB_PASSWORD=getpass("Enter PostgreSQL password: ")
try:
    conn=psycopg2.connect(host=DB_HOST,port=DB_PORT,dbname=DB_NAME,user=DB_USER,password=DB_PASSWORD)
    cursor=conn.cursor()
    print("PostgreSQL connection successful.")
    print("Database:",DB_NAME)
    print("Host:",DB_HOST)
    print("Port:",DB_PORT)
except Exception as e:
    print("PostgreSQL connection failed.")
    print("Error:",e)


PostgreSQL connection successful.
Database: lj_dev_commerce
Host: localhost
Port: 5432


## 6.2 Prepare / Create Target Table

Check whether the approved `commerce.inventory` target table already exists. If it exists, do not recreate it.


In [25]:
target_schema="commerce"
target_table="inventory"
cursor.execute("""SELECT EXISTS (SELECT 1 FROM information_schema.tables WHERE table_schema=%s AND table_name=%s);""",(target_schema,target_table))
table_exists=cursor.fetchone()[0]
print("Inventory Target Table Check")
print("="*80)
print("Schema:",target_schema)
print("Table:",target_table)
print("Table exists:",table_exists)
if table_exists:
    print("\nTarget table already exists.")
    print("The table will not be recreated.")
else:
    print("\nTarget table does not exist.")
    print("The approved database design will be used before creating it.")


Inventory Target Table Check
Schema: commerce
Table: inventory
Table exists: False

Target table does not exist.
The approved database design will be used before creating it.


## 6.2.1 Create Inventory Table

Create `commerce.inventory` only when the target table does not already exist, using the approved Inventory Data Dictionary and foreign-key relationships.


In [26]:
if not table_exists:
    create_inventory_table_query="""
    CREATE TABLE commerce.inventory (
        inventory_id TEXT PRIMARY KEY,
        product_id TEXT NOT NULL,
        warehouse_id TEXT NOT NULL,
        current_stock INTEGER NOT NULL,
        reorder_level INTEGER NOT NULL,
        reserved_stock INTEGER NOT NULL,
        available_stock INTEGER NOT NULL,
        inventory_status TEXT NOT NULL,
        last_stock_update TIMESTAMP NOT NULL,
        created_date TIMESTAMP NOT NULL,
        created_by TEXT NOT NULL,
        updated_date TIMESTAMP NOT NULL,
        updated_by TEXT NOT NULL,
        FOREIGN KEY (product_id) REFERENCES commerce.product(product_id),
        FOREIGN KEY (warehouse_id) REFERENCES commerce.warehouse(warehouse_id)
    );
    """
    try:
        cursor.execute(create_inventory_table_query)
        conn.commit()
        print("Inventory table created successfully.")
    except Exception as e:
        conn.rollback()
        print("Inventory table creation failed.")
        print("Error:",e)
else:
    print("Inventory table already exists. Creation skipped.")


Inventory table created successfully.


## 6.3 Verify Inventory Table Structure

Verify the actual PostgreSQL structure of `commerce.inventory` against the approved target design.


In [27]:
verify_inventory_table_query="""SELECT ordinal_position,column_name,data_type,is_nullable,column_default FROM information_schema.columns WHERE table_schema='commerce' AND table_name='inventory' ORDER BY ordinal_position;"""
try:
    cursor.execute(verify_inventory_table_query)
    table_structure=cursor.fetchall()
    expected_structure=[
        (1,"inventory_id","text","NO"),(2,"product_id","text","NO"),(3,"warehouse_id","text","NO"),
        (4,"current_stock","integer","NO"),(5,"reorder_level","integer","NO"),(6,"reserved_stock","integer","NO"),(7,"available_stock","integer","NO"),
        (8,"inventory_status","text","NO"),(9,"last_stock_update","timestamp without time zone","NO"),(10,"created_date","timestamp without time zone","NO"),
        (11,"created_by","text","NO"),(12,"updated_date","timestamp without time zone","NO"),(13,"updated_by","text","NO")]
    actual_structure=[(row[0],row[1],row[2],row[3]) for row in table_structure]
    structure_valid=actual_structure==expected_structure
    print("Inventory Table Structure")
    print("="*80)
    print(f"{'Position':<10}{'Column Name':<25}{'Data Type':<30}{'Nullable':<12}{'Default'}")
    print("-"*110)
    for row in table_structure:
        print(f"{row[0]:<10}{row[1]:<25}{row[2]:<30}{row[3]:<12}{row[4]}")
    print("\nInventory table structure validation passed:",structure_valid)
    assert structure_valid
except Exception as e:
    print("Inventory table structure verification failed.")
    print("Error:",e)


Inventory Table Structure
Position  Column Name              Data Type                     Nullable    Default
--------------------------------------------------------------------------------------------------------------
1         inventory_id             text                          NO          None
2         product_id               text                          NO          None
3         warehouse_id             text                          NO          None
4         current_stock            integer                       NO          None
5         reorder_level            integer                       NO          None
6         reserved_stock           integer                       NO          None
7         available_stock          integer                       NO          None
8         inventory_status         text                          NO          None
9         last_stock_update        timestamp without time zone   NO          None
10        created_date             times

## 6.4 Verify Keys, Relationships & Load Dependencies

Verify the Inventory primary key, both approved foreign keys, and the required parent records in Product and Warehouse.


In [28]:
print("Inventory Keys, Relationships & Load Dependency Review")
print("="*80)
cursor.execute("""SELECT tc.constraint_name,kcu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='inventory' AND tc.constraint_type='PRIMARY KEY';""")
primary_keys=cursor.fetchall()
primary_key_valid=len(primary_keys)==1 and primary_keys[0][1]=="inventory_id"
print("Primary Key:",primary_keys)
print("Primary Key validation passed:",primary_key_valid)
cursor.execute("""SELECT tc.constraint_name,kcu.column_name,ccu.table_schema,ccu.table_name,ccu.column_name FROM information_schema.table_constraints AS tc JOIN information_schema.key_column_usage AS kcu ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema JOIN information_schema.constraint_column_usage AS ccu ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema WHERE tc.table_schema='commerce' AND tc.table_name='inventory' AND tc.constraint_type='FOREIGN KEY' ORDER BY kcu.column_name;""")
foreign_keys=cursor.fetchall()
expected_foreign_keys={("product_id","product","product_id"),("warehouse_id","warehouse","warehouse_id")}
actual_foreign_keys={(r[1],r[3],r[4]) for r in foreign_keys}
foreign_keys_valid=actual_foreign_keys==expected_foreign_keys
print("Foreign Keys:",foreign_keys)
print("Foreign Key validation passed:",foreign_keys_valid)

product_ids=inventory_db_ready["product_id"].dropna().unique().tolist()
placeholders=",".join(["%s"]*len(product_ids))
cursor.execute(f"SELECT product_id FROM commerce.product WHERE product_id IN ({placeholders});",product_ids)
existing_product_ids={r[0] for r in cursor.fetchall()}
missing_product_ids=set(product_ids)-existing_product_ids
product_dependency_valid=len(missing_product_ids)==0

warehouse_ids=inventory_db_ready["warehouse_id"].dropna().unique().tolist()
placeholders=",".join(["%s"]*len(warehouse_ids))
cursor.execute(f"SELECT warehouse_id FROM commerce.warehouse WHERE warehouse_id IN ({placeholders});",warehouse_ids)
existing_warehouse_ids={r[0] for r in cursor.fetchall()}
missing_warehouse_ids=set(warehouse_ids)-existing_warehouse_ids
warehouse_dependency_valid=len(missing_warehouse_ids)==0

print("Missing Product IDs:",missing_product_ids)
print("Product dependency ready:",product_dependency_valid)
print("Missing Warehouse IDs:",missing_warehouse_ids)
print("Warehouse dependency ready:",warehouse_dependency_valid)
load_dependency_ready=all([primary_key_valid,foreign_keys_valid,product_dependency_valid,warehouse_dependency_valid])
print("Inventory is ready for data loading:",load_dependency_ready)


Inventory Keys, Relationships & Load Dependency Review
Primary Key: [('inventory_pkey', 'inventory_id')]
Primary Key validation passed: True
Foreign Keys: [('inventory_product_id_fkey', 'product_id', 'commerce', 'product', 'product_id'), ('inventory_warehouse_id_fkey', 'warehouse_id', 'commerce', 'warehouse', 'warehouse_id')]
Foreign Key validation passed: True
Missing Product IDs: set()
Product dependency ready: True
Missing Warehouse IDs: set()
Warehouse dependency ready: True
Inventory is ready for data loading: True


## 6.5 Insert Inventory Records

Load the validated Inventory Database-Ready dataset only after target structure and dependency checks pass. Insertion is protected against accidental duplicate loading.


In [29]:
print("Inventory Record Loading")
print("="*80)
cursor.execute("SELECT COUNT(*) FROM commerce.inventory;")
existing_record_count=cursor.fetchone()[0]
target_table_empty=existing_record_count==0
db_ready_row_count=len(inventory_db_ready)
print("Existing records:",existing_record_count)
print("Target table is empty:",target_table_empty)
print("Database-ready records:",db_ready_row_count)
insert_inventory_query="""INSERT INTO commerce.inventory (inventory_id,product_id,warehouse_id,current_stock,reorder_level,reserved_stock,available_stock,inventory_status,last_stock_update,created_date,created_by,updated_date,updated_by) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s);"""
try:
    if not target_table_empty: raise ValueError("Target table is not empty. Insertion stopped to prevent accidental duplicate loading.")
    records_to_insert=[tuple(row) for row in inventory_db_ready.itertuples(index=False,name=None)]
    if len(records_to_insert)!=db_ready_row_count: raise ValueError("Database-ready record count changed before insertion.")
    cursor.executemany(insert_inventory_query,records_to_insert)
    conn.commit()
    print("Inventory records inserted:",len(records_to_insert))
    print("Transaction committed successfully.")
except Exception as e:
    conn.rollback()
    print("Inventory record insertion failed.")
    print("Transaction rolled back.")
    print("Error:",e)


Inventory Record Loading
Existing records: 0
Target table is empty: True
Database-ready records: 19
Inventory records inserted: 19
Transaction committed successfully.


## 6.6 Validate Inventory Row Count

Compare the Database-Ready Inventory row count with the PostgreSQL target row count.


In [30]:
db_ready_row_count=len(inventory_db_ready)
cursor.execute("SELECT COUNT(*) FROM commerce.inventory;")
postgresql_row_count=cursor.fetchone()[0]
row_count_match=db_ready_row_count==postgresql_row_count
print("Inventory Row Count Validation")
print("="*80)
print("\nDatabase-Ready Dataset Row Count:",db_ready_row_count)
print("PostgreSQL Table Row Count:",postgresql_row_count)
print("Row counts match:",row_count_match)
assert row_count_match
print("Inventory row count validation passed.")


Inventory Row Count Validation

Database-Ready Dataset Row Count: 19
PostgreSQL Table Row Count: 19
Row counts match: True
Inventory row count validation passed.


## 6.7 Retrieve Inventory Records

Retrieve the loaded Inventory records from PostgreSQL for direct review.


In [31]:
retrieve_inventory_query="""SELECT inventory_id,product_id,warehouse_id,current_stock,reorder_level,reserved_stock,available_stock,inventory_status,last_stock_update,created_date,created_by,updated_date,updated_by FROM commerce.inventory ORDER BY inventory_id;"""
try:
    cursor.execute(retrieve_inventory_query)
    inventory_records=cursor.fetchall()
    column_names=[description[0] for description in cursor.description]
    inventory_postgresql=pd.DataFrame(inventory_records,columns=column_names)
    for column in ["last_stock_update","created_date","updated_date"]:
        inventory_postgresql[column]=pd.to_datetime(inventory_postgresql[column],errors="coerce")
    print("Inventory Records Retrieved from PostgreSQL")
    print("Records retrieved:",len(inventory_records))
    display(inventory_postgresql)
except Exception as e:
    print("Inventory record retrieval failed.")
    print("Error:",e)


Inventory Records Retrieved from PostgreSQL
Records retrieved: 19


,inventory_id,product_id,warehouse_id,current_stock,reorder_level,reserved_stock,available_stock,inventory_status,last_stock_update,created_date,created_by,updated_date,updated_by
0,IN001,P0001,WH001,25,10,2,23,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
1,IN002,P0002,WH001,30,10,3,27,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
2,IN003,P0003,WH001,12,10,1,11,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
3,IN004,P0004,WH002,40,10,4,36,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
4,IN005,P0005,WH001,18,10,1,17,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
5,IN006,P0006,WH002,22,10,2,20,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
6,IN007,P0007,WH001,16,10,1,15,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
7,IN008,P0008,WH001,20,10,2,18,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
8,IN009,P0009,WH001,35,10,3,32,In Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin
9,IN010,P0010,WH001,10,10,1,9,Low Stock,2026-07-24 18:00:00,2026-07-01,admin,2026-07-24 18:00:00,admin


## 6.8 Source → Database Reconciliation

Compare the Inventory Database-Ready dataset against the PostgreSQL target for row count, columns, primary keys, and record-level values.


In [32]:
inventory_source_reconciliation=inventory_db_ready.copy().sort_values("inventory_id").reset_index(drop=True)
inventory_postgresql_reconciliation=inventory_postgresql.copy().sort_values("inventory_id").reset_index(drop=True)
for column in ["last_stock_update","created_date","updated_date"]:
    inventory_source_reconciliation[column]=pd.to_datetime(inventory_source_reconciliation[column],errors="coerce")
    inventory_postgresql_reconciliation[column]=pd.to_datetime(inventory_postgresql_reconciliation[column],errors="coerce")
source_row_count=len(inventory_source_reconciliation)
database_row_count=len(inventory_postgresql_reconciliation)
row_count_match=source_row_count==database_row_count
source_columns=inventory_source_reconciliation.columns.tolist()
database_columns=inventory_postgresql_reconciliation.columns.tolist()
column_match=source_columns==database_columns
source_inventory_ids=set(inventory_source_reconciliation["inventory_id"])
database_inventory_ids=set(inventory_postgresql_reconciliation["inventory_id"])
primary_key_match=source_inventory_ids==database_inventory_ids
value_match=inventory_source_reconciliation.equals(inventory_postgresql_reconciliation)
reconciliation_passed=all([row_count_match,column_match,primary_key_match,value_match])
print("Inventory Source → Database Reconciliation")
print("Row counts match:",row_count_match)
print("Columns and order match:",column_match)
print("Inventory IDs match:",primary_key_match)
print("All record values match:",value_match)
print("Source → Database reconciliation passed:",reconciliation_passed)
assert reconciliation_passed


Inventory Source → Database Reconciliation
Row counts match: True
Columns and order match: True
Inventory IDs match: True
All record values match: True
Source → Database reconciliation passed: True


## 6.9 Inventory Database Integrity Validation

Perform final Inventory integrity checks covering required fields, primary key uniqueness, foreign-key relationships, quantity business rules, valid status values, and audit date relationships.


In [33]:
integrity_query="""SELECT COUNT(*) AS total_rows, COUNT(DISTINCT inventory_id) AS distinct_inventory_ids, COUNT(*) FILTER (WHERE inventory_id IS NULL) AS null_inventory_ids, COUNT(*) FILTER (WHERE product_id IS NULL) AS null_product_ids, COUNT(*) FILTER (WHERE warehouse_id IS NULL) AS null_warehouse_ids, COUNT(*) FILTER (WHERE current_stock IS NULL) AS null_current_stock, COUNT(*) FILTER (WHERE reorder_level IS NULL) AS null_reorder_level, COUNT(*) FILTER (WHERE reserved_stock IS NULL) AS null_reserved_stock, COUNT(*) FILTER (WHERE available_stock IS NULL) AS null_available_stock, COUNT(*) FILTER (WHERE inventory_status IS NULL) AS null_inventory_statuses, COUNT(*) FILTER (WHERE last_stock_update IS NULL) AS null_last_stock_updates, COUNT(*) FILTER (WHERE created_date IS NULL) AS null_created_dates, COUNT(*) FILTER (WHERE created_by IS NULL) AS null_created_by, COUNT(*) FILTER (WHERE updated_date IS NULL) AS null_updated_dates, COUNT(*) FILTER (WHERE updated_by IS NULL) AS null_updated_by, COUNT(*) FILTER (WHERE current_stock < 0 OR reorder_level < 0 OR reserved_stock < 0 OR available_stock < 0) AS negative_quantity_rows, COUNT(*) FILTER (WHERE available_stock <> current_stock - reserved_stock) AS quantity_mismatches, COUNT(*) FILTER (WHERE inventory_status NOT IN ('In Stock','Low Stock')) AS invalid_status_rows, COUNT(*) FILTER (WHERE updated_date < created_date) AS invalid_audit_date_order, COUNT(*) FILTER (WHERE last_stock_update < created_date) AS invalid_last_update_order FROM commerce.inventory;"""
try:
    cursor.execute(integrity_query)
    r=cursor.fetchone()
    print("Inventory Database Integrity Validation")
    print("="*80)
    labels=["Total rows","Distinct inventory IDs","NULL inventory IDs","NULL product IDs","NULL warehouse IDs","NULL current stock","NULL reorder level","NULL reserved stock","NULL available stock","NULL inventory status","NULL last stock update","NULL created dates","NULL created by","NULL updated dates","NULL updated by","Negative quantity rows","Quantity mismatches","Invalid status rows","Updated before created","Last update before created"]
    for label,value in zip(labels,r): print(f"{label}: {value}")
    integrity_passed=(r[0]==r[1] and all(value==0 for value in r[2:]))
    print("\nInventory database integrity passed:",integrity_passed)
    assert integrity_passed
except Exception as e:
    print("Inventory database integrity validation failed.")
    print("Error:",e)


Inventory Database Integrity Validation
Total rows: 19
Distinct inventory IDs: 19
NULL inventory IDs: 0
NULL product IDs: 0
NULL warehouse IDs: 0
NULL current stock: 0
NULL reorder level: 0
NULL reserved stock: 0
NULL available stock: 0
NULL inventory status: 0
NULL last stock update: 0
NULL created dates: 0
NULL created by: 0
NULL updated dates: 0
NULL updated by: 0
Negative quantity rows: 0
Quantity mismatches: 0
Invalid status rows: 0
Updated before created: 0
Last update before created: 0

Inventory database integrity passed: True


## 6.10 Close PostgreSQL Connection

Close PostgreSQL resources cleanly after all Inventory loading and validation steps are complete.


In [34]:
try:
    if cursor is not None and not cursor.closed: cursor.close()
    if conn is not None and conn.closed == 0: conn.close()
    print("PostgreSQL resources closed successfully.")
    print("Cursor closed:",cursor.closed)
    print("Connection closed:",conn.closed != 0)
except Exception as e:
    print("PostgreSQL resource closure failed.")
    print("Error:",e)


PostgreSQL resources closed successfully.
Cursor closed: True
Connection closed: True
